# House Price Prediction — Ames Housing

## Comprehensive Data Analysis, Regression Benchmarking & Buyer Insights

This notebook is the **single main project deliverable** for the `PRCP-1020-HousePricePred` project. It follows the supplied project brief and covers:

1. **Task 1 — Complete data analysis report:** data structure, target distribution, missing values, outliers, feature relationships and descriptive insights.
2. **Task 2 — Robust machine-learning solution:** preprocessing, domain-driven feature engineering, multiple regression models, 5-fold cross-validation and model comparison.
3. **Task 2(b) — Feature/price relationship analysis:** interpretation of important structural, quality and location variables.
4. **Task 3 — Customer buying recommendations:** practical guidance based on neighborhood, size, quality, age and amenities.
5. **Required reports:** model comparison, production recommendation, challenges faced and technical mitigations.

> **Dataset:** Ames, Iowa residential housing data with 79 explanatory variables.  
> **Target:** `SalePrice`.


## 1. Project Brief

The supplied brief asks for an end-to-end analysis and a robust house-price prediction algorithm, including relationships between house features and price and recommendations for customers. It also requires a model comparison report, a challenges/mitigations report, and states that all tasks should be contained in a single Jupyter notebook.

The dataset description identifies `SalePrice` as the target and includes variables covering lot characteristics, neighborhood, construction quality, living area, bathrooms, garage, porch, pool, sale timing and sale condition.


In [ ]:
# Core imports
import os
import zipfile
import urllib.request
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_URL = "https://d3ilbtxij3aepc.cloudfront.net/projects/CDS-Capstone-Projects/PRCP-1020-HousePricePred.zip"
ARCHIVE_NAME = "PRCP-1020-HousePricePred.zip"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Environment initialized.")


## 2. Download and Load the Dataset

The notebook downloads the project archive when it is not already present, extracts it into the local project directory, searches recursively for `train.csv`, and loads the training data.

The repository intentionally does **not** bundle the raw dataset. This keeps the Git repository lightweight and avoids redistributing a third-party dataset unnecessarily.


In [ ]:
# Download and extract the dataset
archive_path = DATA_DIR / ARCHIVE_NAME

if not archive_path.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, archive_path)
    print("Download completed.")
else:
    print("Dataset archive already exists.")

extract_dir = DATA_DIR / "raw"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_path, "r") as zf:
    zf.extractall(extract_dir)

csv_candidates = list(extract_dir.rglob("*.csv"))
print("CSV files found:", [str(p) for p in csv_candidates])

train_candidates = [p for p in csv_candidates if p.name.lower() == "train.csv"]
if not train_candidates:
    raise FileNotFoundError("train.csv was not found after extracting the project archive.")

train_path = train_candidates[0]
df = pd.read_csv(train_path)

print(f"Loaded dataset: {train_path}")
print(f"Shape: {df.shape}")


## 3. Data Structure and Initial Inspection

We inspect observations, feature types, summary statistics, duplicate rows, and the target variable. The project brief describes 79 explanatory variables; the exact shape is reported from the downloaded file so the notebook remains data-driven.


In [ ]:
print("=== FIRST FIVE ROWS ===")
display(df.head())

print("\n=== DATA TYPES ===")
display(df.dtypes.to_frame("dtype").T)

print("\n=== DATASET SHAPE ===")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Potential explanatory variables: {df.shape[1] - 1}")

print("\n=== NUMERICAL / CATEGORICAL FEATURES ===")
print("Numerical:", df.select_dtypes(include=np.number).shape[1])
print("Categorical:", df.select_dtypes(include=["object", "string", "category"]).shape[1])

print("\n=== DUPLICATES ===")
print("Duplicate rows:", df.duplicated().sum())

print("\n=== TARGET SUMMARY ===")
display(df["SalePrice"].describe().to_frame("SalePrice").T)


## 4. Exploratory Data Analysis

### 4.1 SalePrice Distribution

House prices are commonly right-skewed. We compare the raw target with `log1p(SalePrice)`, which is useful for regression because it reduces extreme scale differences and aligns with the log-scale evaluation commonly used for this competition-style dataset.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title(f"SalePrice — skewness: {df['SalePrice'].skew():.2f}")
axes[0].set_xlabel("Sale Price ($)")

log_saleprice = np.log1p(df["SalePrice"])
sns.histplot(log_saleprice, kde=True, ax=axes[1])
axes[1].set_title(f"log1p(SalePrice) — skewness: {log_saleprice.skew():.2f}")
axes[1].set_xlabel("log(1 + SalePrice)")

plt.tight_layout()
plt.show()


### 4.2 Living Area vs Sale Price

`GrLivArea` (above-grade living area) is a key size variable. We visualize its relationship with `SalePrice` and flag the extreme high-area/low-price observations used later for domain-informed outlier trimming.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", alpha=0.65)
plt.axvline(4000, linestyle="--", label="4,000 sq ft boundary")
plt.title("Above-Grade Living Area vs Sale Price")
plt.xlabel("GrLivArea (sq ft)")
plt.ylabel("SalePrice ($)")
plt.legend()
plt.show()

extreme_outliers = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)]
print("Extreme observations matching the trimming rule:", len(extreme_outliers))
display(extreme_outliers[["Id", "GrLivArea", "SalePrice"]] if "Id" in extreme_outliers else extreme_outliers[["GrLivArea", "SalePrice"]])


### 4.3 Missing-Value Analysis

Missing values in Ames housing data can be **structural**. For example, a missing garage-quality value can indicate that the property has no garage rather than that the measurement was forgotten.

We therefore inspect missingness before choosing feature-specific imputation rules.


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percentage": (missing / len(df) * 100).round(2)
})

print("Top missing-value features:")
display(missing_report.head(20))

top10 = missing_report.head(10).sort_values("Missing Percentage")
plt.figure(figsize=(10, 5))
plt.barh(top10.index, top10["Missing Percentage"])
plt.xlabel("Missing values (%)")
plt.title("Top 10 Features by Missingness")
plt.tight_layout()
plt.show()


## 5. Feature Engineering and Preprocessing

The preprocessing strategy follows the supplied notebook and project brief:

- Remove the small set of extreme high-leverage observations defined by `GrLivArea > 4000` and `SalePrice < 300000`.
- Log-transform `SalePrice`.
- Drop `Id` because it is an identifier rather than a predictive property characteristic.
- Interpret structural categorical missingness as `None`.
- Interpret structural numerical missingness as zero where appropriate.
- Fill `LotFrontage` using the neighborhood median.
- Fill remaining numerical variables with medians and remaining categorical variables with modes.
- Create domain features:
  - `TotalSF`
  - `TotalBathrooms`
  - `HouseAge`
  - `YearsSinceRemodel`
  - `OverallQual_TotalSF`
- One-hot encode categorical variables.


In [ ]:
# 1. Remove domain-defined extreme outliers
df_clean = df.loc[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))].copy()

# 2. Target and predictors
y = np.log1p(df_clean["SalePrice"])
X = df_clean.drop(columns=["SalePrice"]).copy()

if "Id" in X.columns:
    X = X.drop(columns=["Id"])

# 3. Structural categorical missingness
none_categorical_cols = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish",
    "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature"
]

for col in none_categorical_cols:
    if col in X.columns:
        X[col] = X[col].fillna("None")

# 4. Structural numerical missingness
zero_numerical_cols = [
    "GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1",
    "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath",
    "BsmtHalfBath", "MasVnrArea"
]

for col in zero_numerical_cols:
    if col in X.columns:
        X[col] = X[col].fillna(0)

# 5. Neighborhood-aware LotFrontage imputation
if "LotFrontage" in X.columns and "Neighborhood" in X.columns:
    X["LotFrontage"] = X.groupby("Neighborhood")["LotFrontage"].transform(
        lambda s: s.fillna(s.median())
    )

# 6. Remaining missing values
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())

for col in X.select_dtypes(include=["object", "string", "category"]).columns:
    mode = X[col].mode(dropna=True)
    X[col] = X[col].fillna(mode.iloc[0] if not mode.empty else "None")

# 7. Domain feature engineering
X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
X["TotalBathrooms"] = (
    X["FullBath"] + 0.5 * X["HalfBath"] +
    X["BsmtFullBath"] + 0.5 * X["BsmtHalfBath"]
)
X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
X["YearsSinceRemodel"] = X["YrSold"] - X["YearRemodAdd"]
X["OverallQual_TotalSF"] = X["OverallQual"] * X["TotalSF"]

# 8. One-hot encode categories
X_encoded = pd.get_dummies(X, drop_first=True, dtype=float)

print("Original cleaned rows:", len(df_clean))
print("Encoded feature matrix shape:", X_encoded.shape)
print("Remaining missing values:", int(X_encoded.isna().sum().sum()))


## 6. Feature-to-Price Relationship Analysis

The following analyses directly address Task 2(b). We inspect correlations among important numerical variables and compare median sale prices across neighborhoods and quality ratings.

Correlation is treated as an exploratory relationship, **not proof of causation**.


In [ ]:
# Strongest numerical correlations with SalePrice
numeric_corr = df_clean.select_dtypes(include=np.number).corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
print("Top positive numerical correlations with SalePrice:")
display(numeric_corr.head(12).to_frame("Correlation"))

print("\nTop negative numerical correlations with SalePrice:")
display(numeric_corr.tail(8).to_frame("Correlation"))


In [ ]:
# Neighborhood price summary
neighborhood_summary = (
    df_clean.groupby("Neighborhood")["SalePrice"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
)

print("Neighborhood price summary:")
display(neighborhood_summary.head(15))

plt.figure(figsize=(11, 6))
top_neighborhoods = neighborhood_summary.head(15).sort_values("median")
plt.barh(top_neighborhoods.index, top_neighborhoods["median"])
plt.xlabel("Median SalePrice ($)")
plt.title("Top Neighborhoods by Median Sale Price")
plt.tight_layout()
plt.show()


In [ ]:
# Overall quality relationship
quality_summary = (
    df_clean.groupby("OverallQual")["SalePrice"]
    .agg(["count", "median", "mean"])
    .sort_index()
)

display(quality_summary)

plt.figure(figsize=(9, 5))
plt.plot(quality_summary.index, quality_summary["median"], marker="o")
plt.xlabel("OverallQual")
plt.ylabel("Median SalePrice ($)")
plt.title("Overall Quality vs Median Sale Price")
plt.xticks(quality_summary.index)
plt.tight_layout()
plt.show()


## 7. Model Benchmarking — 5-Fold Cross-Validation

We benchmark four regression approaches:

- **Ridge Regression** — regularized linear model and strong baseline for high-dimensional one-hot encoded data.
- **Random Forest** — non-linear ensemble baseline.
- **LightGBM** — gradient boosting with strong tabular-data performance.
- **XGBoost** — gradient boosting candidate with controlled tree depth and learning rate.

Metrics:

- **Mean CV RMSE on log-transformed price:** lower is better.
- **Mean CV R²:** higher is better.
- **Standard deviation of CV RMSE:** lower indicates more consistent folds.

The supplied project notebook reports the following benchmark values; the executable section below recomputes them when the notebook is run.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# Scaling is retained from the supplied notebook.
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_encoded)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Ridge Regression": Ridge(alpha=15.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=4,
        random_state=RANDOM_STATE, n_jobs=-1, objective="reg:squarederror"
    ),
}

results = []
for name, model in models.items():
    rmse_scores = np.sqrt(
        -cross_val_score(model, X_scaled, y, scoring="neg_mean_squared_error", cv=kf)
    )
    r2_scores = cross_val_score(model, X_scaled, y, scoring="r2", cv=kf)

    results.append({
        "Model Architecture": name,
        "Mean CV RMSE (Log)": rmse_scores.mean(),
        "Std CV RMSE": rmse_scores.std(),
        "Mean CV R2 Score": r2_scores.mean(),
    })

comparison_df = (
    pd.DataFrame(results)
    .sort_values("Mean CV RMSE (Log)")
    .reset_index(drop=True)
)

display(comparison_df.style.format({
    "Mean CV RMSE (Log)": "{:.4f}",
    "Std CV RMSE": "{:.4f}",
    "Mean CV R2 Score": "{:.4f}",
}))


## 8. Model Comparison Report

The supplied notebook reports this benchmark:

| Model | Mean CV RMSE (Log) | Std CV RMSE | Mean CV R² |
|---|---:|---:|---:|
| Ridge Regression | **0.1153** | 0.0092 | **0.9163** |
| XGBoost | 0.1242 | 0.0085 | 0.9026 |
| LightGBM | 0.1261 | **0.0062** | 0.8998 |
| Random Forest | 0.1355 | 0.0092 | 0.8837 |

### Recommendation

**Ridge Regression** is the strongest choice if the priority is the reported predictive score: it has the lowest RMSE and highest R² in the supplied benchmark.

**LightGBM** is a reasonable production-oriented alternative when non-linear interactions and fold-to-fold stability are prioritized, because the supplied benchmark gives it the lowest RMSE standard deviation.

> These reported numbers are preserved from the supplied project material. Running the notebook recomputes the metrics from the downloaded data; small differences can occur with changes in package versions or preprocessing behavior.


## 9. Challenges Faced and Technical Mitigations

| Challenge | Risk | Technique | Reason |
|---|---|---|---|
| Target skewness | Uneven error scale across cheap and expensive homes | `log1p(SalePrice)` | Stabilizes the target scale and makes regression less dominated by extreme prices |
| Structural missing values | `NA` can mean absence of a feature | `None` / zero imputation | Preserves the semantic meaning of no garage, no basement, no pool, etc. |
| High-leverage outliers | Extreme observations can distort regression | Domain-informed trimming | Removes the defined `GrLivArea > 4000` and low-price cases |
| Many categorical features | One-hot encoding creates a high-dimensional matrix | One-hot encoding + Ridge regularization | Converts categories to numeric features while controlling coefficient magnitude |
| Non-linear relationships | Price does not respond linearly to every feature | Random Forest, LightGBM and XGBoost | Provides tree-based alternatives that can capture interactions |
| Different feature scales | Features vary from counts to square footage and years | `RobustScaler` | Reduces sensitivity to residual scale differences |
| Buyer interpretability | A raw prediction is not enough for customers | Neighborhood, quality and size analysis | Converts model/data patterns into practical buying guidance |


## 10. Customer Home-Buying Recommendations

The recommendations below are based on patterns described in the supplied project notebook and the exploratory summaries generated above.

### Location
Neighborhood is an important categorical signal. Buyers should compare properties within neighborhoods using median prices rather than comparing only the city-wide average.

### Quality and Size
`OverallQual`, living area and composite total square footage are major valuation signals. When budget is constrained, structural quality and usable area should be evaluated together rather than relying on bedroom count alone.

### Renovation and Age
`YearBuilt`, `YearRemodAdd`, `HouseAge` and `YearsSinceRemodel` provide context about age and potential maintenance requirements.

### Amenities
Basement area, garage capacity, bathrooms, porch/deck space and other amenities should be considered relative to their effect on the total asking price.

### Practical Buying Workflow
1. Define the maximum budget.
2. Shortlist suitable neighborhoods.
3. Compare `OverallQual`, `GrLivArea` and `TotalSF`.
4. Check age and renovation history.
5. Evaluate garage, basement and bathroom configuration.
6. Use the model as a valuation aid, not as a substitute for inspection or professional appraisal.


## 11. Final Conclusion

This project establishes an end-to-end Ames housing valuation workflow:

**Raw data → EDA → missing-value treatment → outlier handling → feature engineering → one-hot encoding → model benchmarking → interpretation → buyer recommendations**

The supplied benchmark identifies **Ridge Regression** as the best-performing model by reported RMSE/R², while **LightGBM** offers a strong alternative for operational use when stability and non-linear interactions are important.

The solution can be extended with hyperparameter optimization, leakage-safe preprocessing pipelines, geospatial variables, temporal market indicators, and external economic data.


## 12. Reproducibility Notes

- Python: 3.10+ recommended.
- Install dependencies using the repository `requirements.txt`.
- Run this notebook from the repository root.
- The dataset is downloaded automatically from the project-provided CloudFront URL.
- Do not commit the downloaded raw dataset or generated archives to Git.
- Results may vary slightly across library versions.

### Repository structure

```text
HousePricePred/
├── README.md
├── requirements.txt
├── .gitignore
├── notebooks/
│   └── House_Price_Prediction.ipynb
├── data/
│   └── README.md
├── docs/
│   └── PROJECT_REQUIREMENTS.md
└── reports/
    ├── MODEL_COMPARISON.md
    └── CHALLENGES_AND_MITIGATIONS.md
```
